# Laboratorio Semana 5 · Transformación de datos y cierre del bloque 1
**Ciencia de datos con Python | Material para aprendices SENA**

**Caso:** seguimiento académico ficticio (continuidad de Semanas 1–4). **Fecha de corte del dataset:** 18/10/2026. Nunca interprete los umbrales didácticos como normativa SENA ni utilice datos personales auténticos.

**Objetivo:** preparar variables para indicadores con transformaciones reproducibles y documentar sus límites.

## 0. Instrucciones de uso
1. Conserve su proyecto y cree `data/processed/` y `outputs/`.
2. Coloque el CSV entregado como `data/processed/seguimiento_academico_semana3_preparado.csv`.
3. Desde la raíz instale `python -m pip install pandas numpy jupyter ipykernel`.
4. Abra este notebook en `notebooks/`; ejecute Kernel → Restart & Run All.
5. No edite el CSV de origen.

In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FUENTE = RAIZ / "data" / "processed" / "seguimiento_academico_semana3_preparado.csv"
SALIDA = RAIZ / "outputs"
PROCESADOS = RAIZ / "data" / "processed"
SALIDA.mkdir(parents=True, exist_ok=True)
PROCESADOS.mkdir(parents=True, exist_ok=True)
assert FUENTE.is_file(), f"No se encontró la fuente: {FUENTE}"

origen = pd.read_csv(FUENTE, dtype={"documento": "string", "ficha": "string"},
                     encoding="utf-8-sig")
df = origen.copy()
for campo in ["nota_python", "nota_datos", "asistencia"]:
    df[campo] = pd.to_numeric(df[campo], errors="coerce")
for campo in ["fecha_actualizacion", "fecha_corte"]:
    df[campo] = pd.to_datetime(df[campo], errors="coerce")
sha256 = hashlib.sha256(FUENTE.read_bytes()).hexdigest()
print("Fuente:", FUENTE.name, "| filas:", len(df))
print("Huella SHA-256:", sha256[:16], "...")
print(df[["documento", "nota_python", "nota_datos", "asistencia"]].head(3))

Fuente: seguimiento_academico_semana3_preparado.csv | filas: 19
Huella SHA-256: 2a7095c80905ff3a ...
  documento  nota_python  nota_datos  asistencia
0      1001          4.2         4.0        92.0
1      1002          3.8         3.6        85.0
2      1003          4.7         4.8        96.0


## 1. Arrastre de calidad desde Semana 3 y Semana 4
Antes de transformar, identifique **claves conflictivas**, **datos ausentes** y **fechas posteriores al corte**.

In [2]:
# Conservar los conflictos sin seleccionar arbitrariamente una versión.
df["conflicto_identidad"] = (
    df["documento"].notna() & df["documento"].duplicated(keep=False)
)
df["fecha_fuera_corte"] = (
    df["fecha_actualizacion"] > df["fecha_corte"]
).fillna(False)
print("Filas:", len(df))
print("Registros en conflicto:", int(df["conflicto_identidad"].sum()))
print("Actualizaciones posteriores al corte:", int(df["fecha_fuera_corte"].sum()))

Filas: 19
Registros en conflicto: 2
Actualizaciones posteriores al corte: 1


## 2. Columnas calculadas y operaciones vectorizadas
`promedio_dos_notas` requiere **ambas** notas; `brecha_asistencia_pp` expresa puntos porcentuales respecto a un umbral exclusivamente didáctico.

In [3]:
# Una nota faltante debe impedir el promedio de dos evaluaciones.
notas = ["nota_python", "nota_datos"]
df["numero_notas"] = df[notas].notna().sum(axis=1)
df["promedio_dos_notas"] = (
    (df["nota_python"] + df["nota_datos"]) / 2
).where(df[notas].notna().all(axis=1))

UMBRAL_ASISTENCIA = 80    # ejemplo didáctico, no regla SENA
UMBRAL_PROMEDIO = 3.5    # ejemplo didáctico, no regla SENA
df["brecha_asistencia_pp"] = df["asistencia"] - UMBRAL_ASISTENCIA
print(df[["documento", "numero_notas", "promedio_dos_notas", "brecha_asistencia_pp"]].head())

  documento  numero_notas  promedio_dos_notas  brecha_asistencia_pp
0      1001             2                4.10                  12.0
1      1002             2                3.70                   5.0
2      1003             2                4.75                  16.0
3      1004             2                3.00                 -10.0
4      1005             2                4.95                  18.0


## 3. Normalización de texto y `replace()`
Conserve el texto fuente; documente el diccionario de equivalencias. Un error como `BELL0` no debe corregirse por adivinanza.

In [4]:
# Conserve columnas originales y cree nuevas variables derivadas.
df["programa_limpio"] = (
    df["programa"].astype("string").str.strip().str.casefold()
    .replace({
        "adso": "ADSO",
        "ciencia datos python": "Ciencia de datos con Python",
        "ciencia de datos con python": "Ciencia de datos con Python",
    })
)
df["ciudad_limpia"] = (
    df["ciudad"].astype("string").str.strip().str.casefold()
    .replace({"medellín": "Medellín", "medellin": "Medellín",
              "bello": "Bello", "itagüí": "Itagüí", "itagui": "Itagüí",
              "envigado": "Envigado"})
)
# 'BELL0' queda sin correspondencia: NO se adivina el dato verdadero.
print(df[["ciudad", "ciudad_limpia"]].tail(9))

      ciudad ciudad_limpia
10  medellín      Medellín
11     BELL0         bell0
12       NaN          <NA>
13  Envigado      Envigado
14    Itagui        Itagüí
15  Medellín      Medellín
16     Bello         Bello
17  medellin      Medellín
18  Medellín      Medellín


## 4. `map()` y códigos de categorías
Revise categorías desconocidas: `map()` las convierte a valores faltantes, mientras `replace()` mantiene sin cambio las no previstas.

In [5]:
# replace() conserva valores sin reemplazo; map() deja NA si no encuentra clave.
codigo_ciudad = {"Medellín": "MED", "Bello": "BEL",
                 "Itagüí": "ITA", "Envigado": "ENV"}
df["codigo_ciudad"] = df["ciudad_limpia"].map(codigo_ciudad)
df["ciudad_por_revisar"] = (
    df["ciudad"].notna() & df["codigo_ciudad"].isna()
)
print(df[["ciudad", "ciudad_limpia", "codigo_ciudad"]].tail(9))
print("Categorías sin equivalencia:",
      df.loc[df["ciudad_por_revisar"], "ciudad"].unique())

      ciudad ciudad_limpia codigo_ciudad
10  medellín      Medellín           MED
11     BELL0         bell0           NaN
12       NaN          <NA>           NaN
13  Envigado      Envigado           ENV
14    Itagui        Itagüí           ITA
15  Medellín      Medellín           MED
16     Bello         Bello           BEL
17  medellin      Medellín           MED
18  Medellín      Medellín           MED
Categorías sin equivalencia: ['BELL0']


## 5. `np.where()` y una regla binaria
Para una clasificación con tres estados, utilice `np.select()` y trate primero los valores desconocidos.

In [6]:
# np.where funciona para una clasificación binaria definida.
df["notas_completas"] = np.where(
    df["numero_notas"].eq(2), "Sí", "No"
)
# np.select distingue el dato ausente de una asistencia baja.
df["estado_asistencia"] = np.select(
    [df["asistencia"].isna(), df["asistencia"].lt(UMBRAL_ASISTENCIA)],
    ["Sin dato", "Bajo umbral didáctico"],
    default="En/sobre umbral didáctico"
)
print(df["estado_asistencia"].value_counts(dropna=False))

estado_asistencia
En/sobre umbral didáctico    14
Sin dato                      3
Bajo umbral didáctico         2
Name: count, dtype: int64


## 6. `np.select()` y precedencia
El orden de las condiciones importa. El primer criterio verdadero determina la categoría. Una etiqueta didáctica es una señal para **revisión humana**, no una decisión disciplinaria.

In [7]:
# IMPORTA el orden: prevalecen conflictos, corte y datos incompletos.
sin_fecha = df["fecha_actualizacion"].isna() | df["fecha_corte"].isna()
incompleto = (df["promedio_dos_notas"].isna() |
              df["asistencia"].isna() | df["documento"].isna())
df["prioridad_revision"] = np.select(
    [
        df["conflicto_identidad"],
        df["fecha_fuera_corte"],
        sin_fecha,
        incompleto,
        (df["promedio_dos_notas"].lt(UMBRAL_PROMEDIO) &
         df["asistencia"].lt(UMBRAL_ASISTENCIA)),
        (df["promedio_dos_notas"].lt(UMBRAL_PROMEDIO) |
         df["asistencia"].lt(UMBRAL_ASISTENCIA)),
    ],
    ["Revisar identidad", "Revisar fecha de corte", "Fecha no verificable",
     "Datos insuficientes", "Prioridad alta", "Prioridad preventiva"],
    default="Seguimiento habitual"
)
print(df["prioridad_revision"].value_counts(dropna=False))

prioridad_revision
Seguimiento habitual      8
Datos insuficientes       4
Prioridad alta            2
Revisar identidad         2
Revisar fecha de corte    1
Fecha no verificable      1
Prioridad preventiva      1
Name: count, dtype: int64


## 7. `apply()` para casos especiales
Compare una explicación fila a fila en una pequeña muestra frente a la clasificación vectorizada aplicada a todo el dataset.

In [8]:
# apply(axis=1): demostración para una explicación de 3 filas.
# Para clasificar masivamente, prefiera operaciones vectorizadas.
def explicar_fila(fila):
    if fila["conflicto_identidad"]:
        return "Identidad duplicada con información por verificar"
    if pd.isna(fila["promedio_dos_notas"]):
        return "Falta una nota: no se calculó el promedio"
    return f"Promedio disponible: {fila['promedio_dos_notas']:.2f}"

muestra = df.head(3).copy()
muestra["explicacion"] = muestra.apply(explicar_fila, axis=1)
print(muestra[["documento", "explicacion"]])

  documento                explicacion
0      1001  Promedio disponible: 4.10
1      1002  Promedio disponible: 3.70
2      1003  Promedio disponible: 4.75


## 8. Pruebas básicas de integridad
Verifique que no se redujo el universo sin justificación, que no se inventaron promedios y que los conflictos siguen identificados.

In [9]:
# Pruebas mínimas: no perder filas ni inventar promedios.
assert len(df) == len(origen)
assert df["promedio_dos_notas"].dropna().between(0, 5).all()
assert df.loc[df["numero_notas"].lt(2), "promedio_dos_notas"].isna().all()
assert df.loc[df["conflicto_identidad"], "prioridad_revision"].eq(
    "Revisar identidad").all()
assert df.loc[df["fecha_fuera_corte"] & ~df["conflicto_identidad"],
              "prioridad_revision"].eq("Revisar fecha de corte").all()
print("Validaciones superadas")

Validaciones superadas


## 9. Indicadores con universo explícito
Reporte indicadores solo después de definir denominador y criterios de exclusión.

In [10]:
# Definir el universo antes de calcular un indicador.
base_analizable = df.loc[
    ~df["conflicto_identidad"] & ~df["fecha_fuera_corte"] &
    df["fecha_actualizacion"].notna() & df["fecha_corte"].notna() &
    df["documento"].notna() & df["promedio_dos_notas"].notna() &
    df["asistencia"].notna()
].copy()

indicadores = pd.DataFrame([
    {"indicador": "Registros preparados", "valor": len(df),
     "universo": "Todas las filas recibidas"},
    {"indicador": "Registros con identidad en conflicto",
     "valor": int(df["conflicto_identidad"].sum()),
     "universo": "Todas las filas recibidas"},
    {"indicador": "Registros aptos para el ejemplo de análisis",
     "valor": len(base_analizable),
     "universo": "Sin conflicto, con fecha válida y variables completas"},
    {"indicador": "Promedio de dos notas (base apta)",
     "valor": round(base_analizable["promedio_dos_notas"].mean(), 2),
     "universo": "Solo registros aptos; no toda la ficha"},
])
indicadores["denominador"] = [len(df), len(df), len(df), len(base_analizable)]
indicadores["numerador"] = [
    len(df), int(df["conflicto_identidad"].sum()), len(base_analizable),
    base_analizable["promedio_dos_notas"].sum(),
]
indicadores["fecha_corte"] = df["fecha_corte"].dropna().drop_duplicates().item()
print(indicadores.to_string(index=False))

                                  indicador  valor                                              universo  denominador  numerador fecha_corte
                       Registros preparados  19.00                             Todas las filas recibidas           19       19.0  2026-10-18
       Registros con identidad en conflicto   2.00                             Todas las filas recibidas           19        2.0  2026-10-18
Registros aptos para el ejemplo de análisis  11.00 Sin conflicto, con fecha válida y variables completas           19       11.0  2026-10-18
          Promedio de dos notas (base apta)   3.99                Solo registros aptos; no toda la ficha           11       43.9  2026-10-18


## 10. Bitácora, exportación y verificación
Guarde datos derivados y la bitácora sin modificar la fuente.

In [11]:
# Conservar el CSV fuente intacto; exportar productos derivados.
bitacora = pd.DataFrame([
    ["T01", "nota_python + nota_datos", "promedio_dos_notas",
     "Promediar solo si ambas notas existen", "Sin dato si falta una"],
    ["T02", "programa, ciudad", "programa_limpio, ciudad_limpia",
     "Normalizar texto con tabla de equivalencias", "No inferir 'BELL0'"],
    ["T03", "documento, fecha, notas, asistencia", "prioridad_revision",
     "np.select con precedencia documentada", "No clasifica personas"],
], columns=["id_regla", "campos_origen", "campos_destino", "regla", "limitacion"])

ruta_df = PROCESADOS / "semana5_seguimiento_transformado.csv"
ruta_ind = SALIDA / "semana5_indicadores.csv"
ruta_reglas = SALIDA / "semana5_bitacora_transformaciones.csv"
df.to_csv(ruta_df, index=False, encoding="utf-8-sig")
indicadores.to_csv(ruta_ind, index=False, encoding="utf-8-sig")
bitacora.to_csv(ruta_reglas, index=False, encoding="utf-8-sig")
verificado = pd.read_csv(ruta_df, dtype={"documento": "string"})
assert verificado["fila_fuente"].tolist() == origen["fila_fuente"].tolist()
assert verificado["archivo_fuente"].equals(origen["archivo_fuente"])
assert pd.to_datetime(verificado["fecha_corte"]).equals(df["fecha_corte"])
assert hashlib.sha256(FUENTE.read_bytes()).hexdigest() == sha256
print("Controles de exportación y fuente superados. SHA-256:", sha256)
print("Productos exportados:", ruta_df.name, ruta_ind.name, ruta_reglas.name)

Controles de exportación y fuente superados. SHA-256: 2a7095c80905ff3abbcbe6be05c356dccdd0bf2d15f30fc714dc08272e724a9d
Productos exportados: semana5_seguimiento_transformado.csv semana5_indicadores.csv semana5_bitacora_transformaciones.csv


## 11. Reto autónomo y cierre del bloque 1
Las respuestas, el diccionario y la sustentación están en [el informe de Semana 5](../docs/Semana_5_Cierre_Bloque_1.md).
El reto se ejecuta desde la raíz con `python -m app.reto_semana5`.